# 06 - Revisión de un caso del frontend

Caso reportado desde el frontend: **50 años, hombre, 95 kg, 169 cm, perímetro abdominal 115 cm, hipertensión diagnosticada sí, fumó sí, bebió alcohol sí, frutas 0 días, ensalada de verduras 0 días → 22 %.**

Objetivo: entender cómo se obtiene ese porcentaje y comprobar que no haya un error técnico. **No se modifica ni se reentrena el modelo congelado** (`models/v2/web_regresion_logistica.joblib`) y los datos 2025 no se usan para seleccionar cambios. Las variaciones de la sección 5 son pruebas de comportamiento del modelo, no ajustes.

In [1]:
import os
import sys
import tempfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from statsmodels.stats.proportion import proportion_confint

from preparacion import DESENLACE, PREDICTORES_WEB, cargar_desarrollo

# Backend real, con una base de datos temporal para no tocar la del proyecto
os.environ['DATABASE_URL'] = f"sqlite:///{Path(tempfile.mkdtemp()) / 'revision.db'}"
os.environ['MODELS_DIR'] = str(Path('../models/v2').resolve())
sys.path.insert(0, str(Path('../backend').resolve()))
from fastapi.testclient import TestClient  # noqa: E402
from app.main import app  # noqa: E402
from app.schemas.diabetes import EvaluacionEntrada  # noqa: E402

pd.set_option('display.float_format', '{:.4f}'.format)
artefacto = joblib.load('../models/v2/web_regresion_logistica.joblib')
pipeline = artefacto['modelo']
print('Modelo:', artefacto['algoritmo'], '| recalibrado:', artefacto['recalibrado'], '| predictores:', artefacto['predictores'])

Modelo: Regresión logística | recalibrado: False | predictores: ['edad', 'hombre', 'imc', 'cintura', 'hta', 'tabaco', 'alcohol', 'frutas_dias', 'verduras_dias']


## 1. Del frontend a la API

El frontend (`frontend/js/app.js`) construye este cuerpo de la solicitud: números con `Number(...)`, respuestas Sí/No como `true`/`false` y los días como enteros. La API valida el cuerpo, calcula el IMC y arma la fila en el orden de predictores del modelo.

In [2]:
payload = {  # exactamente lo que envía app.js para el caso reportado
    'edad': 50, 'sexo': 'masculino', 'peso_kg': 95, 'talla_cm': 169, 'cintura_cm': 115,
    'hipertension': True, 'fuma': True, 'alcohol': True, 'dias_frutas': 0, 'dias_verduras': 0,
    'participante_codigo': None,
}
entrada = EvaluacionEntrada(**payload)
print(f"IMC esperado: 95 / 1.69² = {95 / 1.69 ** 2:.4f}  |  IMC calculado por la API: {entrada.imc}")

with TestClient(app) as cliente:
    respuesta = cliente.post('/predict', json=payload).json()
    modelo_api = app.state.modelo
    fila_api = modelo_api._fila(entrada)

print('Fila que la API entrega al modelo (en el orden de predictores):')
fila_api

IMC esperado: 95 / 1.69² = 33.2621  |  IMC calculado por la API: 33.26
Fila que la API entrega al modelo (en el orden de predictores):


,edad,hombre,imc,cintura,hta,tabaco,alcohol,frutas_dias,verduras_dias
0,50,1.0000,33.2600,115.0000,1.0000,1.0000,1.0000,0,0


## 2. Pipeline guardado frente a la API

In [3]:
fila_directa = pd.DataFrame([{
    'edad': 50, 'hombre': 1.0, 'imc': 95 / 1.69 ** 2, 'cintura': 115, 'hta': 1.0,
    'tabaco': 1.0, 'alcohol': 1.0, 'frutas_dias': 0, 'verduras_dias': 0,
}])[artefacto['predictores']]
p_directa = pipeline.predict_proba(fila_directa)[0, 1]
p_api_exacta = modelo_api.predecir(entrada).probabilidad

print(f'Pipeline guardado (IMC sin redondear): {p_directa:.6f}')
print(f'Clase ModeloRiesgo del backend:        {p_api_exacta:.6f}')
print(f'Respuesta HTTP de la API:              {respuesta["probabilidad"]}  -> se muestra como {round(respuesta["probabilidad"] * 100)} %')

Pipeline guardado (IMC sin redondear): 0.219801
Clase ModeloRiesgo del backend:        0.219826
Respuesta HTTP de la API:              0.2198  -> se muestra como 22 %


## 3. Desglose del cálculo

El modelo es una regresión logística sobre variables estandarizadas (imputación → escalado → regresión logística). No se aplicó recalibración a este modelo (`recalibrado = False`), y no hubo imputación porque el caso no tiene datos faltantes. Para cada predictor:

- **valor estandarizado** *z* = (valor − media) / desviación estándar, con la media y la desviación de los datos de entrenamiento 2021–2024;
- **contribución** = coeficiente estandarizado × *z*, en log-odds.

El log-odds total es el intercepto más la suma de las contribuciones, y la probabilidad es 1 / (1 + e^(−log-odds)).

In [4]:
prep, lr = pipeline.named_steps['prep'], pipeline.named_steps['modelo']
columnas = [n.split('__', 1)[1] for n in prep.get_feature_names_out()]
medias = np.concatenate([prep.named_transformers_[g].named_steps['escalar'].mean_ for g in ('continuas', 'categoricas')])
desvs = np.concatenate([prep.named_transformers_[g].named_steps['escalar'].scale_ for g in ('continuas', 'categoricas')])
z = prep.transform(fila_api)[0]
coef_std = lr.coef_[0]

desglose = pd.DataFrame({
    'predictor': columnas,
    'valor_del_caso': [fila_api.iloc[0][c] for c in columnas],
    'media_referencia': medias,
    'desv_estandar': desvs,
    'z': z,
    'coef_estandarizado': coef_std,
    'contribucion_log_odds': coef_std * z,
    'OR_por_unidad': np.exp(coef_std / desvs),
})
desglose['mostrado_en_pantalla'] = desglose['predictor'].isin(['edad', 'hombre', 'imc', 'cintura', 'hta'])
desglose = desglose.set_index('predictor').loc[artefacto['predictores']]

intercepto = lr.intercept_[0]
log_odds = intercepto + desglose['contribucion_log_odds'].sum()
probabilidad = 1 / (1 + np.exp(-log_odds))
display(desglose)
print(f'Intercepto (log-odds de la persona promedio): {intercepto:+.4f}  -> probabilidad {1 / (1 + np.exp(-intercepto)):.4f}')
print(f'Suma de contribuciones:                       {desglose["contribucion_log_odds"].sum():+.4f}')
print(f'Log-odds total:                               {log_odds:+.4f}')
print(f'Probabilidad reconstruida:                    {probabilidad:.6f}  (pipeline: {p_directa:.6f})')

,valor_del_caso,media_referencia,desv_estandar,z,coef_estandarizado,contribucion_log_odds,OR_por_unidad,mostrado_en_pantalla
predictor,,,,,,,,
edad,50.0000,43.4712,10.9341,0.5971,0.6485,0.3872,1.0611,True
hombre,1.0000,0.4528,0.4978,1.0993,-0.1103,-0.1212,0.8013,True
imc,33.2600,29.1051,4.9942,0.8319,-0.3462,-0.2880,0.9330,True
cintura,115.0000,96.5731,11.2698,1.6351,0.5947,0.9724,1.0542,True
hta,1.0000,0.0992,0.2990,3.0130,0.3334,1.0046,3.0503,True
tabaco,1.0000,0.1780,0.3825,2.1488,0.1123,0.2414,1.3414,False
alcohol,1.0000,0.7709,0.4202,0.5451,-0.0836,-0.0456,0.8195,False
frutas_dias,0.0000,4.0322,2.4923,-1.6179,0.0338,-0.0546,1.0136,False
verduras_dias,0.0000,2.8001,2.3248,-1.2045,0.1765,-0.2125,1.0789,False


Intercepto (log-odds de la persona promedio): -3.1503  -> probabilidad 0.0411
Suma de contribuciones:                       +1.8836
Log-odds total:                               -1.2667
Probabilidad reconstruida:                    0.219826  (pipeline: 0.219801)


In [5]:
grupos = {'edad': 'Edad', 'hombre': 'Sexo', 'imc': 'Composición corporal', 'cintura': 'Composición corporal',
          'hta': 'Hipertensión', 'tabaco': 'Tabaco (oculto)', 'alcohol': 'Alcohol (oculto)',
          'frutas_dias': 'Frutas (oculto)', 'verduras_dias': 'Verduras (oculto)'}
por_factor = desglose.groupby(desglose.index.map(grupos))['contribucion_log_odds'].sum().sort_values(key=abs, ascending=False)
print('Contribuciones agrupadas como en la API (log-odds):')
por_factor.to_frame()

Contribuciones agrupadas como en la API (log-odds):


,contribucion_log_odds
predictor,
Hipertensión,1.0046
Composición corporal,0.6844
Edad,0.3872
Tabaco (oculto),0.2414
Verduras (oculto),-0.2125
Sexo,-0.1212
Frutas (oculto),-0.0546
Alcohol (oculto),-0.0456


## 4. Cómo leer las contribuciones (SHAP en un modelo lineal)

En una regresión logística con variables estandarizadas, la contribución de cada variable es exactamente su valor SHAP en la escala de log-odds, tomando como **referencia la persona promedio** de 2021–2024 (z = 0 en todas las variables). Esa persona promedio tiene un log-odds igual al intercepto. Las contribuciones indican **cuánto se aleja este caso de esa persona promedio dentro del modelo**, y suman exactamente la diferencia de log-odds.

Esto no equivale a efectos causales: describen cómo el modelo combina los datos, no qué pasaría con la probabilidad real de tener diabetes si una persona cambiara un hábito. Las contribuciones son aditivas en log-odds, pero no en puntos porcentuales: el mismo aumento de log-odds sube más la probabilidad cuanto más alta es la probabilidad de partida.

## 5. Pruebas de comportamiento: una variable a la vez

Con el modelo congelado, se cambia una sola variable del caso y se mantiene el resto. Para las medidas corporales se cambia por separado el peso (que modifica el IMC) y el perímetro abdominal, y luego ambos.

In [6]:
def probabilidad_de(**cambios):
    datos = {**payload, **cambios}
    return modelo_api.predecir(EvaluacionEntrada(**datos)).probabilidad


base = probabilidad_de()
escenarios = [
    ('Caso reportado', {}),
    ('Edad 30', {'edad': 30}), ('Edad 40', {'edad': 40}), ('Edad 60', {'edad': 60}), ('Edad 69', {'edad': 69}),
    ('Sexo: mujer', {'sexo': 'femenino'}),
    ('Peso 71 kg (IMC 24.9), misma cintura', {'peso_kg': 71}),
    ('Peso 120 kg (IMC 42.0), misma cintura', {'peso_kg': 120}),
    ('Cintura 94 cm, mismo peso', {'cintura_cm': 94}),
    ('Cintura 130 cm, mismo peso', {'cintura_cm': 130}),
    ('Peso 71 kg y cintura 90 cm', {'peso_kg': 71, 'cintura_cm': 90}),
    ('Sin hipertensión', {'hipertension': False}),
    ('No fuma', {'fuma': False}),
    ('No bebe alcohol', {'alcohol': False}),
    ('Frutas 7 días', {'dias_frutas': 7}),
    ('Ensalada de verduras 7 días', {'dias_verduras': 7}),
]
comportamiento = pd.DataFrame([
    {'escenario': nombre, 'probabilidad_%': 100 * probabilidad_de(**cambios),
     'cambio_puntos_%': 100 * (probabilidad_de(**cambios) - base)}
    for nombre, cambios in escenarios
])
comportamiento

,escenario,probabilidad_%,cambio_puntos_%
0,Caso reportado,21.9826,0.0000
1,Edad 30,7.9223,-14.0603
2,Edad 40,13.4725,-8.5101
3,Edad 60,33.7704,11.7877
4,Edad 69,46.5126,24.5299
5,Sexo: mujer,26.0164,4.0338
6,"Peso 71 kg (IMC 24.9), misma cintura",33.5274,11.5447
7,"Peso 120 kg (IMC 42.0), misma cintura",13.3093,-8.6733
8,"Cintura 94 cm, mismo peso",8.5112,-13.4715
9,"Cintura 130 cm, mismo peso",38.3401,16.3575


## 6. ¿Qué tan confiables son las probabilidades cercanas al 22 %?

Se revisa la calibración disponible, sin usar 2025 para decidir cambios:

- **Desarrollo 2021–2024:** predicciones fuera de partición con la misma especificación del modelo congelado (mismo pipeline e hiperparámetros, 5 particiones estratificadas, semilla 42), agrupadas por rangos de probabilidad predicha, con intervalos de confianza al 95% (Wilson) para la proporción observada.
- **Evaluación 2025:** la tabla de deciles ya calculada en el notebook 04, a la que solo se agregan los intervalos de confianza.

In [7]:
d = cargar_desarrollo()
y = d[DESENLACE].astype(int)
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = cross_val_predict(clone(pipeline), d[PREDICTORES_WEB], y, cv=folds, method='predict_proba')[:, 1]

rangos = [0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 1.0]
tabla = pd.DataFrame({'p': oof, 'y': y.to_numpy(), 'rango': pd.cut(oof, rangos, right=False)})
cal_dev = tabla.groupby('rango', observed=True).agg(n=('y', 'size'), casos=('y', 'sum'), predicha_media=('p', 'mean'), observada=('y', 'mean'))
ic = proportion_confint(cal_dev['casos'], cal_dev['n'], method='wilson')
cal_dev['IC95_inf'], cal_dev['IC95_sup'] = ic
print(f"Registros con probabilidad fuera de partición entre 0.20 y 0.25: {((oof >= 0.20) & (oof < 0.25)).sum()} de {len(oof)}")
cal_dev

Registros con probabilidad fuera de partición entre 0.20 y 0.25: 173 de 8909


,n,casos,predicha_media,observada,IC95_inf,IC95_sup
rango,,,,,,
"[0.0, 0.05)",5839,139,0.0256,0.0238,0.0202,0.0280
"[0.05, 0.1)",1553,134,0.0705,0.0863,0.0733,0.1013
"[0.1, 0.15)",678,73,0.1212,0.1077,0.0865,0.1333
"[0.15, 0.2)",280,52,0.1720,0.1857,0.1445,0.2354
"[0.2, 0.25)",173,33,0.2232,0.1908,0.1392,0.2558
"[0.25, 0.3)",116,39,0.2743,0.3362,0.2567,0.4262
"[0.3, 0.4)",188,61,0.3438,0.3245,0.2616,0.3943
"[0.4, 1.0)",82,31,0.4598,0.3780,0.2808,0.4862


In [8]:
cal_2025 = pd.read_csv('../resultados/04_evaluacion/calibracion_deciles_web.csv')
cal_2025['n'] = 199  # 1,990 registros en 10 deciles
cal_2025['casos'] = (cal_2025['observada'] * cal_2025['n']).round().astype(int)
ic = proportion_confint(cal_2025['casos'], cal_2025['n'], method='wilson')
cal_2025['IC95_inf'], cal_2025['IC95_sup'] = ic
cal_2025

,decil,predicha,observada,n,casos,IC95_inf,IC95_sup
0,0,0.0121,0.0151,199,3,0.0051,0.0434
1,1,0.0170,0.0101,199,2,0.0028,0.0359
2,2,0.0212,0.0151,199,3,0.0051,0.0434
3,3,0.0253,0.0151,199,3,0.0051,0.0434
4,4,0.0305,0.0352,199,7,0.0171,0.0708
5,5,0.0383,0.0352,199,7,0.0171,0.0708
6,6,0.0507,0.0553,199,11,0.0311,0.0963
7,7,0.0743,0.0905,199,18,0.0580,0.1384
8,8,0.1171,0.1256,199,25,0.0866,0.1789
9,9,0.2651,0.2864,199,57,0.2281,0.3528


## 7. Conclusión

**No hay error técnico.** El 22 % corresponde al comportamiento aprendido por el modelo congelado.

- **Entrada y transformaciones correctas.** El frontend envía los valores en las unidades esperadas (kg, cm, Sí/No como verdadero/falso, días como enteros). La API calcula un IMC de 33.26 (95 / 1.69²) y arma la fila en el orden de predictores del modelo.
- **La API coincide con el pipeline guardado:** 0.21983 (API) frente a 0.21980 (pipeline con el IMC sin redondear); la diferencia se debe al redondeo del IMC a dos decimales. No hay recalibración ni imputación en este caso.
- **El cálculo se reconstruye exactamente.** La persona promedio de 2021–2024 tiene un log-odds de −3.150 (probabilidad de 4.1 %). Las contribuciones del caso suman +1.884, de modo que el log-odds total es −1.267 y la probabilidad, 1 / (1 + e^1.267) = 0.2198. Las contribuciones principales son la hipertensión diagnosticada (+1.00), la composición corporal (+0.68; el perímetro abdominal +0.97 y el IMC −0.29) y la edad (+0.39). Entre las variables que no se muestran en pantalla, el tabaco suma +0.24 y la ensalada de verduras −0.21.

**Comportamientos del modelo que conviene conocer** (sección 5; no son errores, sino lo que el modelo aprendió de los datos):

- **IMC y perímetro abdominal deben leerse juntos.** A igual perímetro abdominal, un IMC menor *aumenta* la estimación (con 71 kg sube a 34 %) y uno mayor la reduce (con 120 kg baja a 13 %), porque el modelo estima el efecto de cada medida manteniendo fija la otra (r = 0.90). Combinaciones como un IMC de 24.9 con 115 cm de perímetro abdominal son poco frecuentes en los datos. Cuando ambas medidas bajan juntas (71 kg y 90 cm), la estimación baja a 12 %.
- **Las variables ocultas en pantalla también influyen.** Comer ensalada de verduras los 7 días sube la estimación a 32 %, y no beber alcohol, a 26 %. Son asociaciones observadas en los datos, cuya causa no puede determinarse (para el alcohol, ver el notebook 05; las verduras no se han analizado con el mismo detalle). No deben interpretarse como efectos de esos hábitos.

**Calibración cerca del 22 %.**

- En desarrollo (predicciones fuera de partición), 173 de 8,909 registros tuvieron probabilidades entre 20 % y 25 %. La predicción media fue de 22.3 % y la proporción observada, de 19.1 % (IC 95 %: 13.9 %–25.6 %). Es compatible con una calibración adecuada, pero la incertidumbre es amplia (alrededor de ±6 puntos).
- En 2025, el decil superior agrupa todas las probabilidades altas (predicción media 26.5 %, observada 28.6 %; IC 95 %: 22.8 %–35.3 %). Es coherente, pero demasiado grueso para evaluar específicamente el 22 %.
- En consecuencia, el 22 % debe leerse como una estimación aproximada (un rango probable de unos 15 % a 25 %), no como un valor exacto.

**Qué significa el 22 %.** Entre las personas de 30 a 69 años de Lima Metropolitana encuestadas por la ENDES con un perfil como este, aproximadamente 22 de cada 100 **reportaron haber recibido un diagnóstico médico de diabetes**, frente a 6.3 de cada 100 en la población de referencia. No es la probabilidad de tener diabetes: las personas con diabetes no diagnosticada figuran como negativas en los datos, por lo que la probabilidad real de tener la enfermedad podría ser distinta (probablemente mayor), y el modelo no puede estimarla. Tampoco es un riesgo de desarrollar diabetes en el futuro, porque los datos son transversales.